In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# Load your dataset
df = pd.read_csv("set2&3_features.csv")

# Separate features and target
X = df.drop(columns=["RUL(min)"])   # Features
y = df["RUL(min)"]                  # Target

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
# Step 1: Random Forest for feature transformation
rf = RandomForestRegressor(n_estimators=300, random_state=42)

# Train RF
rf.fit(X_train, y_train)

# Transform features using RF (leaf indices)
X_train_rf = rf.apply(X_train)
X_test_rf = rf.apply(X_test)

In [5]:
scaler = StandardScaler()

# Scale RF output
X_train_rf = scaler.fit_transform(X_train_rf)
X_test_rf = scaler.transform(X_test_rf)

In [6]:
param_grid = {
    "C": [10, 50, 100],
    "gamma": ["scale", 0.01, 0.001],
    "epsilon": [0.1, 1, 10]
}

grid = GridSearchCV(
    SVR(kernel="rbf"),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train_rf, y_train)

svm = grid.best_estimator_

In [7]:
y_pred = svm.predict(X_test_rf)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 7212.101608933068
R2 Score: 0.6700913864182274


---

# Feature Extraction

In [8]:
def extract_features(signal):
    signal = np.array(signal)
    features = {}

    # basic
    mean = np.mean(signal)
    std = np.std(signal)
    rms = np.sqrt(np.mean(signal**2))
    peak = np.max(np.abs(signal))

    features["mean"] = mean
    features["std"] = std
    features["rms"] = rms
    features["max"] = np.max(signal)
    features["min"] = np.min(signal)
    features["peak_to_peak"] = np.ptp(signal)

    # statistical
    features["skewness"] = pd.Series(signal).skew()
    features["kurtosis"] = pd.Series(signal).kurt()

    # vibration specific
    mean_abs = np.mean(np.abs(signal))
    sqrt_mean = np.mean(np.sqrt(np.abs(signal)))

    features["crest_factor"] = peak / rms if rms != 0 else 0
    features["shape_factor"] = rms / mean_abs if mean_abs != 0 else 0
    features["impulse_factor"] = peak / mean_abs if mean_abs != 0 else 0
    features["clearance_factor"] = peak / (sqrt_mean**2) if sqrt_mean != 0 else 0

    # entropy
    hist, _ = np.histogram(signal, bins=50, density=True)
    hist = hist + 1e-12
    features["entropy"] = -np.sum(hist * np.log(hist))

    # frequency features
    fft_vals = np.abs(np.fft.fft(signal))
    fft_vals = fft_vals[:len(fft_vals)//2]

    features["fft_mean"] = np.mean(fft_vals)
    features["fft_std"] = np.std(fft_vals)
    features["fft_max"] = np.max(fft_vals)

    # energy
    features["energy"] = np.sum(signal**2)
    features["log_energy"] = np.sum(np.log(signal**2 + 1e-12))

    return pd.DataFrame([features])

In [9]:
def predict_rul_from_raw(signal):
    # Step 1: Extract features
    features = extract_features(signal)

    # Step 2: Align columns with training data
    features = features[X.columns]

    # Step 3: RF transformation
    features_rf = rf.apply(features)

    # Step 4: Scaling
    features_rf = scaler.transform(features_rf)

    # Step 5: Prediction
    rul = svm.predict(features_rf)

    return rul[0]

In [ ]:
from pathlib import Path

signal_path = Path("/kaggle/input/datasets/akashraj77/signal-array")

if signal_path.is_dir():
    signal_files = sorted(
        p for p in signal_path.rglob("*")
        if p.is_file() and p.suffix.lower() in {".txt", ".csv", ".dat"}
    )
    if not signal_files:
        raise FileNotFoundError(f"No readable signal file found inside {signal_path}")
    signal_path = signal_files[0]

new_signal = np.loadtxt(signal_path, dtype=float, encoding="utf-8-sig")

predicted_rul = predict_rul_from_raw(new_signal)

print(f"Loaded {new_signal.size} samples from {signal_path.name}")
print("Predicted RUL in minutes:", predicted_rul)

Predicted RUL in minutes: 2060.7467197017286


In [11]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df)

             Feature  Importance
13          fft_mean    0.708531
7           kurtosis    0.073884
1                std    0.070460
2                rms    0.022360
16            energy    0.019286
0               mean    0.018735
15           fft_max    0.015463
9       shape_factor    0.014499
6           skewness    0.012142
14           fft_std    0.009135
4                min    0.006879
3                max    0.006440
12           entropy    0.006438
5       peak_to_peak    0.004571
8       crest_factor    0.004008
11  clearance_factor    0.003703
10    impulse_factor    0.003464
